# 08 · Declarative pipelines
*Describe what each table should be and let the platform work out the how.*

The pipeline code lives in `pipelines/shield_insight.py` (open it first). Decorators like `@dp.table` only work
inside a **pipeline**, so this notebook is the companion: create the pipeline, run it, then inspect results.

### Create the pipeline (UI)
1. **Jobs & Pipelines → Create → ETL pipeline**.
2. Name `shield_insight`, default catalog `shield`, default schema `pipeline`
   (create it first: `CREATE SCHEMA IF NOT EXISTS shield.pipeline`).
3. Choose **Add existing assets** → root folder = this repo's `pipelines` folder, source file `shield_insight.py`.
4. Compute: **serverless**. Add configuration `raw_path = /Volumes/shield/landing/raw`.
5. **Run pipeline**. Then deliver more files with notebook 03's `deliver()` and run it again: only the new files are processed.

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS shield.pipeline")
display(spark.sql("SHOW TABLES IN shield.pipeline"))

## Inspect results

In [ ]:
display(spark.sql("SELECT severity, COUNT(*) AS n FROM shield.pipeline.incidents_silver GROUP BY severity"))

In [ ]:
display(spark.sql("SELECT * FROM shield.pipeline.daily_threat_briefing ORDER BY report_date DESC LIMIT 10"))

## Data quality metrics live in the pipeline event log
In the pipeline UI open `incidents_silver` → **Data quality** tab, or query the event log. Replace the table
name below with your pipeline's event log (pipeline settings → *Advanced* → event log, or the
`event_log(TABLE(...))` function).

In [ ]:
display(
    spark.sql(
        """
        SELECT timestamp, details:flow_progress.data_quality.expectations AS expectations
        FROM event_log(TABLE(shield.pipeline.incidents_silver))
        WHERE event_type = 'flow_progress' AND details:flow_progress.data_quality IS NOT NULL
        ORDER BY timestamp DESC
        LIMIT 5
        """
    )
)

## Try it yourself
1. Change `expect_or_drop` to `expect` on one rule and compare the silver row counts.
2. Rewrite `daily_threat_briefing` in SQL (`CREATE MATERIALIZED VIEW ...`) in a second file.
3. Add a quarantine table to the pipeline: a table with the inverse of the rules.
4. What happens to a materialized view when you **Full refresh** vs. a normal run?